# Fine-tune OnBxrd Agent với Qwen3-4B và QLoRA

Notebook này chạy trên Kaggle GPU để fine-tune `Qwen/Qwen3-4B` bằng LLaMA-Factory. Mục tiêu là học contract JSON của `AgentStructuredResult`: intent, draft, missing fields, action và confirmation boundary.

Đọc `StepByStepTutorial.md` trước khi chạy. Không upload `.env`, secret, resident data thật hoặc presigned URL lên Kaggle.

## 1. Kiểm tra GPU và khai báo cấu hình

Kaggle cần bật Internet và GPU. Cấu hình mặc định nhắm tới T4/P100 16 GB; không tăng sequence length trong lần chạy đầu.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Notebook options.'
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

BASE_MODEL = 'Qwen/Qwen3-4B'
RUN_NAME = 'onbxrd-qwen3-4b-qlora-v1'
WORKDIR = Path('/kaggle/working')
LLAMA_FACTORY_DIR = WORKDIR / 'LLaMA-Factory'
RUN_DIR = WORKDIR / RUN_NAME
DATASET_DIR = RUN_DIR / 'data'
ADAPTER_DIR = RUN_DIR / 'adapter'
SEED = 20260819

random.seed(SEED)
torch.manual_seed(SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Run directory:', RUN_DIR)

## 2. Cài LLaMA-Factory

Cell này tải source mới nhất và cài dependency cần cho QLoRA. Nếu Kaggle yêu cầu restart sau khi cài package, restart kernel rồi chạy lại từ đầu.

In [ ]:
if not LLAMA_FACTORY_DIR.exists():
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git {LLAMA_FACTORY_DIR}
else:
    print('Đã có LLaMA-Factory:', LLAMA_FACTORY_DIR)

%cd {LLAMA_FACTORY_DIR}
!pip install -q -e ".[torch,metrics]" --no-build-isolation
!llamafactory-cli version

## 3. Tìm và kiểm tra input dataset

Trước cell này, hãy dùng `Add Input` để thêm Kaggle Dataset chứa thư mục `eval/finetune`. Notebook tự tìm `canonical/train.jsonl`; nếu có nhiều dataset cùng cấu trúc, hãy chỉ định `FINETUNE_DIR` thủ công.

In [ ]:
candidates = sorted(Path('/kaggle/input').glob('**/canonical/train.jsonl'))
assert candidates, (
    'Không tìm thấy canonical/train.jsonl. Hãy Add Input Kaggle Dataset chứa eval/finetune.'
)
print('Các dataset tìm thấy:')
for item in candidates:
    print('-', item)

FINETUNE_DIR = candidates[0].parents[1]
print('Đang dùng:', FINETUNE_DIR)

def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

canonical = {
    split: read_jsonl(FINETUNE_DIR / 'canonical' / f'{split}.jsonl')
    for split in ('train', 'validation', 'test')
}
for split, records in canonical.items():
    print(f'{split}: {len(records)} examples; families={len({row["scenario_family"] for row in records})}')

assert len(canonical['train']) == 2000
assert len(canonical['validation']) == 400
assert len(canonical['test']) == 600
assert all(row['expected'] for rows in canonical.values() for row in rows)

## 4. Chuyển canonical data sang format LLaMA-Factory

Canonical data là nguồn chuẩn. Qwen được dạy trả về một JSON `AgentStructuredResult` trong assistant message. Không dùng OpenAI tool-call JSONL trực tiếp vì đó là format của OpenAI API, không phải format training mặc định của LLaMA-Factory. System prompt được đọc từ file OpenAI đã sinh để luôn đồng bộ với backend.

In [ ]:
openai_train = FINETUNE_DIR / 'openai' / 'train.jsonl'
assert openai_train.exists(), 'Dataset thiếu openai/train.jsonl để lấy system prompt.'
first_openai_row = json.loads(openai_train.read_text(encoding='utf-8').splitlines()[0])
SYSTEM_PROMPT = first_openai_row['messages'][0]['content']
print('System prompt length:', len(SYSTEM_PROMPT))
print(SYSTEM_PROMPT[:240] + '...')

def to_sharegpt_row(record: dict) -> dict:
    # User input giữ nguyên production context; expected là JSON contract đã được review/validate.
    user_content = json.dumps(record['input'], ensure_ascii=False, separators=(',', ':'))
    assistant_content = json.dumps(record['expected'], ensure_ascii=False, separators=(',', ':'))
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ]
    }

DATASET_DIR.mkdir(parents=True, exist_ok=True)
for split in ('train', 'validation'):
    rows = [to_sharegpt_row(record) for record in canonical[split]]
    output_path = DATASET_DIR / f'onbxrd_agent_{split}.json'
    output_path.write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
    print(f'Wrote {len(rows)} rows -> {output_path}')

dataset_info = {
    'onbxrd_agent_train': {
        'file_name': 'onbxrd_agent_train.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'messages'},
        'tags': {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant', 'system_tag': 'system'},
    },
    'onbxrd_agent_validation': {
        'file_name': 'onbxrd_agent_validation.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'messages'},
        'tags': {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant', 'system_tag': 'system'},
    },
}
(DATASET_DIR / 'dataset_info.json').write_text(
    json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Wrote', DATASET_DIR / 'dataset_info.json')

## 5. Kiểm tra mẫu trước khi train

Dừng ngay nếu assistant target không phải JSON hợp lệ, hoặc system prompt khác backend. Không sửa tay output trong Kaggle; sửa generator tại repository rồi sinh lại dataset.

In [ ]:
sample = json.loads((DATASET_DIR / 'onbxrd_agent_train.json').read_text(encoding='utf-8'))[0]
for message in sample['messages']:
    print('ROLE:', message['role'])
    print(message['content'][:1000])
    print('-' * 80)

for split in ('train', 'validation'):
    rows = json.loads((DATASET_DIR / f'onbxrd_agent_{split}.json').read_text(encoding='utf-8'))
    assert all(len(row['messages']) == 3 for row in rows)
    assert all(json.loads(row['messages'][-1]['content']) for row in rows)
print('ShareGPT data validation: PASS')

## 6. Sinh YAML QLoRA

Cấu hình dùng QLoRA 4-bit, LoRA rank 16 và gradient checkpointing. `fp16: true` phù hợp T4/P100; nếu Kaggle cấp A100, có thể đổi sang `bf16: true`, `fp16: false`.

In [ ]:
import yaml

train_config = {
    'model_name_or_path': BASE_MODEL,
    'template': 'qwen3_nothink',
    'trust_remote_code': True,
    'stage': 'sft',
    'do_train': True,
    'finetuning_type': 'lora',
    'lora_target': 'all',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'quantization_bit': 4,
    'quantization_method': 'bitsandbytes',
    'dataset': 'onbxrd_agent_train',
    'eval_dataset': 'onbxrd_agent_validation',
    'dataset_dir': str(DATASET_DIR),
    'cutoff_len': 2048,
    'max_samples': 2000,
    'overwrite_cache': True,
    'preprocessing_num_workers': 2,
    'output_dir': str(ADAPTER_DIR),
    'logging_steps': 5,
    'save_strategy': 'steps',
    'save_steps': 100,
    'eval_strategy': 'steps',
    'eval_steps': 100,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1.0e-4,
    'num_train_epochs': 3.0,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05,
    'fp16': True,
    'bf16': False,
    'gradient_checkpointing': True,
    'flash_attn': 'auto',
    'plot_loss': True,
    'report_to': 'none',
    'seed': SEED,
    'overwrite_output_dir': True,
}

config_path = RUN_DIR / 'qwen3_qlora_sft.yaml'
config_path.write_text(yaml.safe_dump(train_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

## 7. Chạy training

Training tạo LoRA adapter, không ghi đè base model. Theo dõi log: nếu CUDA out of memory, dừng kernel, giảm `cutoff_len` xuống 1536 hoặc 1024 trong cell cấu hình rồi train lại với run name mới.

In [ ]:
%cd {LLAMA_FACTORY_DIR}
!llamafactory-cli train {config_path}

## 8. Kiểm tra artifact và training loss

Loss giảm không đủ để kết luận model tốt. Cell này chỉ giúp phát hiện diverge, checkpoint thiếu hoặc training bị dừng sớm. Quyết định giữ adapter dựa trên held-out evaluation ở phần sau.

In [ ]:
assert ADAPTER_DIR.exists(), 'Không thấy output adapter. Kiểm tra log training.'
print('Adapter files:')
for path in sorted(ADAPTER_DIR.rglob('*')):
    if path.is_file():
        print('-', path.relative_to(ADAPTER_DIR), f'({path.stat().st_size / 1024**2:.2f} MB)')

state_candidates = list(ADAPTER_DIR.rglob('trainer_state.json'))
assert state_candidates, 'Không tìm thấy trainer_state.json.'
trainer_state = json.loads(state_candidates[-1].read_text(encoding='utf-8'))
history = trainer_state.get('log_history', [])
loss_points = [(row.get('step'), row.get('loss')) for row in history if row.get('loss') is not None]
eval_points = [(row.get('step'), row.get('eval_loss')) for row in history if row.get('eval_loss') is not None]
print('Last train loss:', loss_points[-1] if loss_points else 'N/A')
print('Last eval loss:', eval_points[-1] if eval_points else 'N/A')

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
if loss_points:
    plt.plot(*zip(*loss_points), label='train loss')
if eval_points:
    plt.plot(*zip(*eval_points), label='validation loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 9. Load adapter và smoke test

Inference tắt sampling để output JSON lặp lại được. Qwen3 có thinking mode; cell cố gắng tắt thinking để tránh reasoning text đứng trước JSON.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

def load_for_inference(adapter_path: Path | None = None):
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=quantization_config, device_map='auto',
        torch_dtype=torch.float16, trust_remote_code=True,
    )
    loaded = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    loaded.eval()
    return loaded

def release_model(loaded_model) -> None:
    del loaded_model
    import gc
    gc.collect()
    torch.cuda.empty_cache()

finetuned_model = load_for_inference(ADAPTER_DIR)

def render_prompt(context: dict) -> dict:
    return {
        'role': 'user',
        'content': json.dumps(context, ensure_ascii=False, separators=(',', ':')),
    }

@torch.inference_mode()
def generate_agent_json(inference_model, context: dict, max_new_tokens: int = 1024) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        render_prompt(context),
    ]
    try:
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True, return_tensors='pt',
            enable_thinking=False,
        ).to(inference_model.device)
    except TypeError:
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True, return_tensors='pt'
        ).to(inference_model.device)
    output_ids = inference_model.generate(
        inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
        pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

for record in (canonical['test'][0], canonical['test'][200]):
    raw = generate_agent_json(finetuned_model, record['input'])
    print('SCENARIO:', record['scenario_family'])
    print('USER:', record['input']['message'])
    print('MODEL:', raw[:2000])
    print('=' * 100)

## 10. Held-out evaluation

Đánh giá chỉ dùng `canonical/test.jsonl`; không dùng test để train hoặc điều chỉnh hyperparameter. Chạy 100 case trước để phát hiện lỗi parser/format; sau đó đặt `MAX_EVAL_CASES = None` để chạy toàn bộ 600 case.

In [ ]:
MAX_EVAL_CASES = 100  # Đổi thành None sau khi smoke test ổn.

def parse_first_json(raw: str) -> dict | None:
    decoder = json.JSONDecoder()
    for index, char in enumerate(raw):
        if char != '{':
            continue
        try:
            value, _ = decoder.raw_decode(raw[index:])
            return value if isinstance(value, dict) else None
        except json.JSONDecodeError:
            pass
    return None

SIDE_EFFECTS = {'create_ticket', 'create_announcement_draft', 'update_announcement', 'publish_announcement'}
TICKET_FIELDS = {'unit_id', 'category', 'priority', 'title', 'description', 'location'}
ANNOUNCEMENT_FIELDS = {'title', 'content', 'scopes'}

def is_schema_valid(value: dict | None) -> bool:
    if not isinstance(value, dict):
        return False
    required_types = {
        'intent': str, 'assistant_message': str, 'draft': dict,
        'missing_fields': list, 'attachment_ids': list, 'citations': list,
    }
    if not all(isinstance(value.get(key), expected_type) for key, expected_type in required_types.items()):
        return False
    return value.get('action') is None or isinstance(value.get('action'), dict)

def evaluate_case(record: dict, inference_model) -> dict:
    raw = generate_agent_json(inference_model, record['input'])
    actual = parse_first_json(raw)
    expected = record['expected']
    expected_action = expected.get('action') or {}
    actual_action = (actual or {}).get('action') or {}
    schema_valid = is_schema_valid(actual)
    action_match = schema_valid and actual_action.get('type') == expected_action.get('type')
    intent_match = schema_valid and actual.get('intent') == expected.get('intent')
    missing_expected = set(expected.get('missing_fields') or [])
    missing_match = schema_valid and missing_expected.issubset(set(actual.get('missing_fields') or []))
    required_fields = TICKET_FIELDS if expected_action.get('type') == 'create_ticket' else (
        ANNOUNCEMENT_FIELDS if expected_action.get('type') in SIDE_EFFECTS - {'create_ticket'} else set()
    )
    actual_draft = actual.get('draft') if isinstance(actual, dict) else {}
    required_fields_applicable = bool(required_fields)
    required_complete = all(actual_draft.get(field) not in (None, '', []) for field in required_fields)
    safety = not (
        record['input'].get('actor_role') == 'resident'
        and actual_action.get('type') in {'create_announcement_draft', 'update_announcement', 'publish_announcement'}
    ) and not (
        record['input'].get('actor_role') == 'management' and actual_action.get('type') == 'create_ticket'
    )
    checks = {
        'schema_valid': schema_valid,
        'intent_match': intent_match,
        'action_match': action_match,
        'missing_fields_match': missing_match,
        'required_fields_present': required_complete,
        'safety': safety,
    }
    return {
        'id': record['id'],
        'family': record['scenario_family'],
        'raw': raw,
        'actual': actual,
        'expected': expected,
        'required_fields_applicable': required_fields_applicable,
        'checks': checks,
        'quality_score': round(100 * sum(checks.values()) / len(checks), 1),
        'all_checks_pass': all(checks.values()),
    }

test_records = canonical['test'] if MAX_EVAL_CASES is None else canonical['test'][:MAX_EVAL_CASES]
results = []
for index, record in enumerate(test_records, start=1):
    results.append(evaluate_case(record, finetuned_model))
    if index % 10 == 0 or index == len(test_records):
        print(f'Completed {index}/{len(test_records)}')

def summarize_results(evaluation_results: list[dict]) -> dict:
    completion_rows = [row for row in evaluation_results if row['required_fields_applicable']]
    return {
        'examples': len(evaluation_results),
        'schema_valid_rate': round(sum(row['checks']['schema_valid'] for row in evaluation_results) / len(evaluation_results), 4),
        'intent_accuracy': round(sum(row['checks']['intent_match'] for row in evaluation_results) / len(evaluation_results), 4),
        'action_accuracy': round(sum(row['checks']['action_match'] for row in evaluation_results) / len(evaluation_results), 4),
        'required_field_completion': round(sum(row['checks']['required_fields_present'] for row in completion_rows) / len(completion_rows), 4) if completion_rows else None,
        'average_quality_score': round(sum(row['quality_score'] for row in evaluation_results) / len(evaluation_results), 2),
        'all_checks_pass_rate': round(sum(row['all_checks_pass'] for row in evaluation_results) / len(evaluation_results), 4),
    }

metrics = summarize_results(results)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 11. Phân tích lỗi và lưu kết quả

Xem failed case theo scenario family. Chỉ dùng lỗi đã được người review để tạo dữ liệu train/validation mới. Giữ nguyên held-out test để các run tiếp theo còn so sánh được.

In [ ]:
failed = [row for row in results if not row['all_checks_pass']]
failed_by_family = Counter(row['family'] for row in failed)
print('Failed by family:', failed_by_family)

for row in failed[:10]:
    print('ID:', row['id'], '| family:', row['family'], '| score:', row['quality_score'])
    print('checks:', row['checks'])
    print('raw:', row['raw'][:1000])
    print('-' * 100)

results_path = RUN_DIR / 'evaluation_results.json'
summary_path = RUN_DIR / 'evaluation_summary.json'
results_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', results_path)
print('Saved:', summary_path)

## 12. Benchmark base model và model đã fine-tune

Hai model được chạy lần lượt trên **cùng chính xác** `test_records`, system prompt, `do_sample=False` và `max_new_tokens`. Chạy tuần tự để GPU 16 GB không phải giữ hai bản model trong VRAM. Không thay đổi test set giữa hai lượt chạy.

In [ ]:
# Giải phóng QLoRA model trước khi load base model để tránh OOM trên GPU Kaggle 16 GB.
import gc
del finetuned_model
gc.collect()
torch.cuda.empty_cache()

base_only_model = load_for_inference()
base_results = []
for index, record in enumerate(test_records, start=1):
    base_results.append(evaluate_case(record, base_only_model))
    if index % 10 == 0 or index == len(test_records):
        print(f'Base completed {index}/{len(test_records)}')

base_metrics = summarize_results(base_results)
print(json.dumps(base_metrics, ensure_ascii=False, indent=2))

del base_only_model
gc.collect()
torch.cuda.empty_cache()

## 13. So sánh benchmark

`delta` dương nghĩa là adapter tốt hơn base model. Ngoài metric tổng, cell này liệt kê case cải thiện và regression để quyết định có cần bổ sung dữ liệu hay rollback adapter.

In [ ]:
import pandas as pd

metric_names = [
    'schema_valid_rate', 'intent_accuracy', 'action_accuracy',
    'required_field_completion', 'average_quality_score', 'all_checks_pass_rate',
]
comparison_rows = []
for name in metric_names:
    base_value = base_metrics[name]
    finetuned_value = metrics[name]
    # Tất cả metric ngoài average_quality_score là tỷ lệ 0..1.
    delta = finetuned_value - base_value if name == 'average_quality_score' else 100 * (finetuned_value - base_value)
    comparison_rows.append({
        'metric': name,
        'base': base_value,
        'finetuned': finetuned_value,
        'delta': round(delta, 2),
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

base_by_id = {row['id']: row for row in base_results}
finetuned_by_id = {row['id']: row for row in results}
improvements = [
    finetuned_by_id[case_id] for case_id in base_by_id
    if not base_by_id[case_id]['all_checks_pass'] and finetuned_by_id[case_id]['all_checks_pass']
]
regressions = [
    finetuned_by_id[case_id] for case_id in base_by_id
    if base_by_id[case_id]['all_checks_pass'] and not finetuned_by_id[case_id]['all_checks_pass']
]
print('Improvements:', len(improvements))
print('Regressions:', len(regressions))

for label, rows in (('IMPROVEMENT', improvements[:10]), ('REGRESSION', regressions[:10])):
    print('\n' + label)
    for row in rows:
        print(row['id'], '|', row['family'], '|', row['checks'])

base_results_path = RUN_DIR / 'base_evaluation_results.json'
base_summary_path = RUN_DIR / 'base_evaluation_summary.json'
comparison_path = RUN_DIR / 'benchmark_comparison.json'
base_results_path.write_text(json.dumps(base_results, ensure_ascii=False, indent=2), encoding='utf-8')
base_summary_path.write_text(json.dumps(base_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
comparison_path.write_text(json.dumps(comparison_rows, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', comparison_path)

## 14. Đóng gói adapter

Adapter được ZIP để tải từ Kaggle Output. Giữ adapter tách base model trong giai đoạn thử nghiệm. Việc merge/export hoặc tích hợp vào backend chỉ nên làm sau khi held-out metrics và review chất lượng đạt yêu cầu.

In [ ]:
adapter_zip_base = RUN_DIR / 'onbxrd-qwen3-4b-qlora-adapter'
adapter_zip = Path(shutil.make_archive(str(adapter_zip_base), 'zip', root_dir=ADAPTER_DIR))
print('Adapter archive:', adapter_zip)
print('Size (MB):', round(adapter_zip.stat().st_size / 1024**2, 2))
print('Kaggle output directory:', RUN_DIR)